# Project Data Analysis

EDA on the **same slices** the indexer uses from `backend/indexer/data/`:

- **`*_data.csv`** — tabular fields plus `ABSTRACT_TEXT` (use these for pandas profiling, plots, and text-length stats).
- **`*_data_embedded.ndjson`** — one JSON object per line: same fields plus an **`embedding`** array (384 floats). These files are what you bulk-import into OpenSearch via `import_embeddings.py` inside Docker; the notebook reads them **directly from disk** for vector stats — no OpenSearch connection required for basic EDA.

**Docker / OpenSearch** (for the live index, not required for file-based EDA): delete index, import NDJSONs, verify count:

```bash
curl -s -X DELETE "http://localhost:9200/project_data"
docker compose exec backend python indexer/import_embeddings.py \
  /app/2020_data_embedded.ndjson ... /app/2025_data_embedded.ndjson
curl -s "http://localhost:9200/project_data/_count"
```

Run this notebook with working directory **`backend/indexer`** so paths like `data/2025_data.csv` resolve correctly.

In [29]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

DATA_DIR = Path("data")


def load_year_csvs_from_data_dir(years: list[int] | None = None) -> pd.DataFrame:
    """Load `data/{year}_data.csv` files (embedding pipeline / OpenSearch document shape without vectors)."""
    if years is None:
        paths = sorted(DATA_DIR.glob("*_data.csv"))
    else:
        paths = [DATA_DIR / f"{y}_data.csv" for y in years]
    paths = [p for p in paths if p.exists()]
    if not paths:
        return pd.DataFrame()
    frames: list[pd.DataFrame] = []
    for p in paths:
        year = int(p.name.split("_")[0])
        chunk = pd.read_csv(p)
        chunk["SOURCE_YEAR"] = year
        frames.append(chunk)
    return pd.concat(frames, ignore_index=True)


# Prefer canonical CSVs under backend/indexer/data/
df_project = load_year_csvs_from_data_dir()
df_abstract: pd.DataFrame | None = None

if df_project.empty:
    DATA_PATH = Path("Project cv's/2025_ProjectData.csv")
    DATA_PATH2 = Path("Project cv's/2025abs_ProjectData.csv")
    if not DATA_PATH.exists():
        raise FileNotFoundError(
            f"No *_data.csv under {DATA_DIR.resolve()} and legacy export missing: {DATA_PATH}"
        )
    df_project = pd.read_csv(DATA_PATH)
    df_abstract = pd.read_csv(DATA_PATH2)

print(f"Loaded {len(df_project):,} rows × {len(df_project.columns)} columns")
df_project.info()

KeyboardInterrupt: 

### Optional: embedding vectors (NDJSON)

The `*_data_embedded.ndjson` files are large. Sample the first *n* lines to check dimensions and norms without loading everything into RAM. Full-document analytics still use the CSV cells above.

In [ ]:
import json

import numpy as np

NDJSON_PATH = DATA_DIR / "2025_data_embedded.ndjson"
SAMPLE_LINES = 500

if NDJSON_PATH.exists():
    rows = []
    with NDJSON_PATH.open(encoding="utf-8") as fh:
        for i, line in enumerate(fh):
            if i >= SAMPLE_LINES:
                break
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    emb = np.asarray([r["embedding"] for r in rows], dtype=np.float64)
    norms = np.linalg.norm(emb, axis=1)
    print(f"Sampled {len(rows)} docs from {NDJSON_PATH.name}")
    print(f"embedding shape: {emb.shape}  |  mean L2 norm: {norms.mean():.4f}")
else:
    print(f"Skip: no file at {NDJSON_PATH}")

In [ ]:
# Column names and data types
df_project.info()

<class 'pandas.DataFrame'>
RangeIndex: 75995 entries, 0 to 75994
Data columns (total 46 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   APPLICATION_ID             75995 non-null  int64  
 1   ACTIVITY                   75995 non-null  str    
 2   ADMINISTERING_IC           75995 non-null  str    
 3   APPLICATION_TYPE           74923 non-null  float64
 4   ARRA_FUNDED                75995 non-null  str    
 5   AWARD_NOTICE_DATE          72018 non-null  str    
 6   BUDGET_START               72019 non-null  str    
 7   BUDGET_END                 71996 non-null  str    
 8   ASSISTANCE_LISTING_NUMBER  61597 non-null  float64
 9   CORE_PROJECT_NUM           75995 non-null  str    
 10  ED_INST_TYPE               46925 non-null  str    
 11  OPPORTUNITY NUMBER         71995 non-null  str    
 12  FULL_PROJECT_NUM           75995 non-null  str    
 13  FUNDING_ICs                74068 non-null  str    
 14  F

In [4]:
# Keep only selected columns
keep_cols = [
    "APPLICATION_ID",
    "PROJECT_TITLE",
    "PI_NAMEs",
    "ORG_NAME",
    "ORG_CITY",
    "ORG_STATE",
    "ORG_ZIPCODE",
    "ORG_COUNTRY",
    "IC_NAME",
    "FY",
    "PROJECT_START",
    "PROJECT_END",
    "TOTAL_COST",
    "ACTIVITY",
    "PROJECT_TERMS",
    "ABSTRACT_TEXT",
    "SOURCE_YEAR",
]

missing_cols = [c for c in keep_cols if c not in df_project.columns]
if missing_cols:
    print("These requested columns were not found:", missing_cols)

df_project = df_project[[c for c in keep_cols if c in df_project.columns]].copy()
print(f"DataFrame shape after column filter: {df_project.shape}")
df_project.head()

DataFrame shape after column filter: (75995, 15)


,APPLICATION_ID,PROJECT_TITLE,PI_NAMEs,ORG_NAME,ORG_CITY,ORG_STATE,ORG_ZIPCODE,ORG_COUNTRY,IC_NAME,FY,PROJECT_START,PROJECT_END,TOTAL_COST,ACTIVITY,PROJECT_TERMS
0,9339833,HSR&D Research Career Scientist Award Application,"TIMKO, CHRISTINE D (contact)",VETERANS ADMIN PALO ALTO HEALTH CARE SYS,PALO ALTO,CA,943041207,UNITED STATES,Veterans Affairs,2025,2017-04-01,2024-03-31,NaN,IK6,Acute;addiction;Address;Aftercare;alcohol abus...
1,9339957,HSR&D Research Career Scientist Award Application,"HUMPHREYS, KEITH N. (contact)",VETERANS ADMIN PALO ALTO HEALTH CARE SYS,PALO ALTO,CA,943041207,UNITED STATES,Veterans Affairs,2025,2017-04-01,2024-03-31,NaN,IK6,addiction;Address;Age;Alcohol abuse;Alcohol co...
2,9340481,HSR&D Research Career Scientist Award Application,"WAGNER, TODD H (contact)",VETERANS ADMIN PALO ALTO HEALTH CARE SYS,PALO ALTO,CA,943041207,UNITED STATES,Veterans Affairs,2025,2017-04-01,2022-03-31,NaN,IK6,Accountability;Address;Adverse effects;Affect;...
3,9341478,HSR&D Research Career Scientist Award Application,"FORTNEY, JOHN C. (contact)",VA PUGET SOUND HEALTHCARE SYSTEM,SEATTLE,WA,981081532,UNITED STATES,Veterans Affairs,2025,2017-07-01,2022-06-30,NaN,IK6,Address;Age;Ambulatory Care Facilities;Area;Aw...
4,9773483,HSR&D Senior Research Career Scientist Award,"HYNES, DENISE M. (contact)",PORTLAND VA MEDICAL CENTER,PORTLAND,OR,972392964,UNITED STATES,Veterans Affairs,2025,2019-04-01,2026-03-31,NaN,IK6,Advisory Committees;Anemia;Appointment;Attenti...


In [5]:
if df_abstract is not None:
    df_merged = pd.merge(
        df_project,
        df_abstract,
        on="APPLICATION_ID",
        how="left",
    )
else:
    # data/*.csv already includes ABSTRACT_TEXT in the same row
    df_merged = df_project.copy()

print(f"Rows: {len(df_merged):,}")
df_merged.head()


Matched rows: 75,995


,APPLICATION_ID,PROJECT_TITLE,PI_NAMEs,ORG_NAME,ORG_CITY,ORG_STATE,ORG_ZIPCODE,ORG_COUNTRY,IC_NAME,FY,PROJECT_START,PROJECT_END,TOTAL_COST,ACTIVITY,PROJECT_TERMS,ABSTRACT_TEXT
0,9339833,HSR&D Research Career Scientist Award Application,"TIMKO, CHRISTINE D (contact)",VETERANS ADMIN PALO ALTO HEALTH CARE SYS,PALO ALTO,CA,943041207,UNITED STATES,Veterans Affairs,2025,2017-04-01,2024-03-31,NaN,IK6,Acute;addiction;Address;Aftercare;alcohol abus...,Renewal of my VA HSR&D Senior Research Career ...
1,9339957,HSR&D Research Career Scientist Award Application,"HUMPHREYS, KEITH N. (contact)",VETERANS ADMIN PALO ALTO HEALTH CARE SYS,PALO ALTO,CA,943041207,UNITED STATES,Veterans Affairs,2025,2017-04-01,2024-03-31,NaN,IK6,addiction;Address;Age;Alcohol abuse;Alcohol co...,Dr. Humphreys is a Senior Research Career Scie...
2,9340481,HSR&D Research Career Scientist Award Application,"WAGNER, TODD H (contact)",VETERANS ADMIN PALO ALTO HEALTH CARE SYS,PALO ALTO,CA,943041207,UNITED STATES,Veterans Affairs,2025,2017-04-01,2022-03-31,NaN,IK6,Accountability;Address;Adverse effects;Affect;...,Dr. Wagner is a highly productive VA health ec...
3,9341478,HSR&D Research Career Scientist Award Application,"FORTNEY, JOHN C. (contact)",VA PUGET SOUND HEALTHCARE SYSTEM,SEATTLE,WA,981081532,UNITED STATES,Veterans Affairs,2025,2017-07-01,2022-06-30,NaN,IK6,Address;Age;Ambulatory Care Facilities;Area;Aw...,This Research Career Scientist award is for Dr...
4,9773483,HSR&D Senior Research Career Scientist Award,"HYNES, DENISE M. (contact)",PORTLAND VA MEDICAL CENTER,PORTLAND,OR,972392964,UNITED STATES,Veterans Affairs,2025,2019-04-01,2026-03-31,NaN,IK6,Advisory Committees;Anemia;Appointment;Attenti...,This application for a Senior Research Career ...


### Quick EDA (tabular)

Use `df_merged` for distributions, missingness, and text-length stats. Extend with `groupby`, `pivot_table`, and seaborn as needed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if "TOTAL_COST" in df_merged.columns:
    sns.histplot(df_merged["TOTAL_COST"].dropna(), bins=40, ax=axes[0])
    axes[0].set_title("TOTAL_COST distribution")
if "FY" in df_merged.columns:
    df_merged["FY"].value_counts().sort_index().plot(kind="bar", ax=axes[1], rot=0)
    axes[1].set_title("Projects by FY")
plt.tight_layout()
plt.show()

if "ABSTRACT_TEXT" in df_merged.columns:
    lens = df_merged["ABSTRACT_TEXT"].astype(str).str.len()
    plt.figure(figsize=(8, 4))
    sns.histplot(lens, bins=40)
    plt.title("Abstract character length")
    plt.show()

### Unique `PROJECT_TERMS` from **`backend/indexer/data/`**

NIH-style cells often list many terms **separated by semicolons (`;`)**.

The next cell resolves **`backend/indexer/`** (same as locating `proj_data_analysis.ipynb`), then reads **only** the CSV exports in **`data/`** — files named **`{year}_data.csv`** (the same slice used for embeddings / OpenSearch). It streams **`PROJECT_TERMS`** in chunks, splits on **`;`**, and collects every distinct token.

Output: **`unique_PROJECT_TERMS_tokens.txt`** next to this notebook.

If your files use another delimiter, change `SPLIT_ON` in the code cell.

In [28]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display


def resolve_indexer_dir() -> Path:
    """Folder that contains this notebook — works when kernel cwd is repo root, `backend/`, or `indexer/`."""
    cwd = Path.cwd().resolve()
    candidates: list[Path] = [cwd, *cwd.parents]
    for root in (cwd, cwd.parent):
        candidates.extend(
            [
                root / "backend" / "indexer",
                root / "indexer",
            ],
        )
    seen: set[Path] = set()
    for base in candidates:
        try:
            b = base.resolve()
        except OSError:
            continue
        if b in seen:
            continue
        seen.add(b)
        if (b / "proj_data_analysis.ipynb").is_file():
            return b
    return cwd


def discover_terms_column(columns: list[str]) -> str | None:
    upper = {c.upper(): c for c in columns}
    return upper.get("PROJECT_TERMS")


# --- config: canonical CVs live under backend/indexer/data/ ---
INDEXER = resolve_indexer_dir()
DATA_DIR = INDEXER / "data"
OUT_FILE = INDEXER / "unique_PROJECT_TERMS_tokens.txt"
SPLIT_ON = ";"
CHUNK_ROWS = 50_000
PREVIEW_ROWS = 200

cwd = Path.cwd().resolve()
paths = sorted(p.resolve() for p in DATA_DIR.glob("*_data.csv") if p.is_file())

print(f"Indexer dir (notebook): {INDEXER}")
print(f"Kernel cwd: {cwd}")
print(f"Data dir (CVs): {DATA_DIR}")
print("Discovered CSV paths:", len(paths))
for p in paths:
    print(" ", p)

if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        f"Expected CV folder missing:\n  {DATA_DIR.resolve()}\n"
        "Create `backend/indexer/data/` and add `{year}_data.csv` exports (e.g. `2025_data.csv`).",
    )

if not paths:
    extra = ""
    try:
        other = sorted(DATA_DIR.glob("*.csv"))
        if other:
            extra = "\nCSV files present in data/ (no *_data.csv match):\n" + "\n".join(
                f"  • {p.name}" for p in other[:30]
            )
            if len(other) > 30:
                extra += f"\n  … and {len(other) - 30} more"
    except OSError:
        pass
    raise FileNotFoundError(
        f"No `*_data.csv` files under:\n  {DATA_DIR.resolve()}\n"
        "Add files like `2020_data.csv`, `2021_data.csv`, … matching your pipeline exports."
        + extra,
    )

term_set: set[str] = set()
for path in paths:
    print(f"Scanning {path} …")
    header = pd.read_csv(path, nrows=0, on_bad_lines="skip")
    terms_col = discover_terms_column(list(header.columns))
    if terms_col is None:
        print(f"  skip (no PROJECT_TERMS column; columns: {list(header.columns)[:12]}…)")
        continue
    try:
        reader = pd.read_csv(
            path,
            usecols=[terms_col],
            chunksize=CHUNK_ROWS,
            dtype=str,
            low_memory=False,
            on_bad_lines="skip",
        )
    except Exception as exc:
        print(f"  skip (read error): {exc}")
        continue

    for chunk in reader:
        col = chunk[terms_col].dropna()
        for raw in col.astype(str):
            for piece in raw.split(SPLIT_ON):
                t = piece.strip()
                if t:
                    term_set.add(t)

sorted_terms = sorted(term_set)
print(f"\nFiles scanned: {len(paths)}  |  Unique tokens (split on {SPLIT_ON!r}): {len(sorted_terms):,}\n")

OUT_FILE.write_text("\n".join(sorted_terms) + "\n", encoding="utf-8")
print(f"Wrote full list ({len(sorted_terms):,} lines) → {OUT_FILE.resolve()}\n")

if not sorted_terms:
    print("No terms collected (empty column or all blank).")
else:
    preview = pd.DataFrame({"PROJECT_TERM_TOKEN": sorted_terms[:PREVIEW_ROWS]})
    display(preview)
    if len(sorted_terms) > PREVIEW_ROWS:
        print(f"… preview shows first {PREVIEW_ROWS} rows only; see file for complete list.")

Indexer dir (notebook): /Users/Student/Documents/VSCODE/Sprinternship/backend/indexer
Kernel cwd: /Users/Student/Documents/VSCODE/Sprinternship/backend/indexer
Data dir (CVs): /Users/Student/Documents/VSCODE/Sprinternship/backend/indexer/data
Discovered CSV paths: 6
  /Users/Student/Documents/VSCODE/Sprinternship/backend/indexer/data/2020_data.csv
  /Users/Student/Documents/VSCODE/Sprinternship/backend/indexer/data/2021_data.csv
  /Users/Student/Documents/VSCODE/Sprinternship/backend/indexer/data/2022_data.csv
  /Users/Student/Documents/VSCODE/Sprinternship/backend/indexer/data/2023_data.csv
  /Users/Student/Documents/VSCODE/Sprinternship/backend/indexer/data/2024_data.csv
  /Users/Student/Documents/VSCODE/Sprinternship/backend/indexer/data/2025_data.csv
Scanning /Users/Student/Documents/VSCODE/Sprinternship/backend/indexer/data/2020_data.csv …
Scanning /Users/Student/Documents/VSCODE/Sprinternship/backend/indexer/data/2021_data.csv …
Scanning /Users/Student/Documents/VSCODE/Sprinterns

,PROJECT_TERM_TOKEN
0,",3&apos"
1,",4&apos"
2,",5&apos"
3,-AMP-activated protein kinase
4,-Cyclic-Nucleotide Phosphodiesterases
...,...
195,4-Aminobutyrate aminotransferase
196,4-Aminopyridine
197,4-Hydroxy-Tamoxifen
198,4-Nitroquinoline-1-oxide


… preview shows first 200 rows only; see file for complete list.


### Umbrella themes from `PROJECT_TERMS` → word cloud

**Embeddings note:** each row has **one** vector for *title + `PROJECT_TERMS` + abstract* together. There are **no separate vectors** per `;`-split term from that pipeline.

**Thematic word cloud:** split `PROJECT_TERMS` on `;` as before, then map each piece to an **umbrella label** (e.g. *blood*, *nutrition* → **Health**) using **substring rules** in `UMBRELLA_KEYWORDS`. Count how often each umbrella appears and draw a cloud.

Edit the keyword tuples for your science domain. First matching umbrella wins (dict order). Install once: `pip install wordcloud`.

In [30]:
from __future__ import annotations

from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# --- resolve data/ (same idea as PROJECT_TERMS cell) ---
def resolve_indexer_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates: list[Path] = [cwd, *cwd.parents]
    for root in (cwd, cwd.parent):
        candidates.extend(
            [
                root / "backend" / "indexer",
                root / "indexer",
            ],
        )
    seen: set[Path] = set()
    for base in candidates:
        try:
            b = base.resolve()
        except OSError:
            continue
        if b in seen:
            continue
        seen.add(b)
        if (b / "proj_data_analysis.ipynb").is_file():
            return b
    return cwd


INDEXER = resolve_indexer_dir()
DATA_DIR = INDEXER / "data"
SPLIT_ON = ";"
CHUNK_ROWS = 50_000

# Map semicolon-split terms → umbrella words (substring match, case-insensitive).
# Put more specific umbrellas before generic ones if you add overlapping keywords.
UMBRELLA_KEYWORDS: dict[str, tuple[str, ...]] = {
    "Health": (
        "blood",
        "wellbeing",
        "well-being",
        "nutrition",
        "diabetes",
        "obesity",
        "cardio",
        "heart",
        "mental health",
        "cancer",
        "immune",
        "vaccin",
        "pathogen",
        "syndrome",
        "patient",
        "clinical",
        "therapy",
        "tumor",
        "metabolic",
        "exercise",
        "sleep",
        "stress",
    ),
    "Technology": (
        "algorithm",
        "software",
        "machine learning",
        "neural",
        "computing",
        "database",
        "sensor",
        "robot",
        "imaging",
        "device",
        "gpu",
        "simulation",
    ),
    "Education": (
        "student",
        "school",
        "curriculum",
        "learning",
        "training",
        "workshop",
        "undergraduate",
        "mentor",
    ),
}


def umbrella_label(term: str) -> str:
    lower = term.lower().strip()
    if not lower:
        return "Other"
    for label, needles in UMBRELLA_KEYWORDS.items():
        if any(n in lower for n in needles):
            return label
    return "Other"


try:
    from wordcloud import WordCloud
except ImportError as exc:
    raise ImportError("Run: pip install wordcloud") from exc

paths = sorted(p.resolve() for p in DATA_DIR.glob("*_data.csv") if p.is_file())
if not paths:
    raise FileNotFoundError(f"No *_data.csv under {DATA_DIR.resolve()}")

counts: Counter[str] = Counter()
for path in paths:
    for chunk in pd.read_csv(
        path,
        usecols=["PROJECT_TERMS"],
        chunksize=CHUNK_ROWS,
        dtype=str,
        low_memory=False,
        on_bad_lines="skip",
    ):
        for raw in chunk["PROJECT_TERMS"].dropna():
            for piece in str(raw).split(SPLIT_ON):
                t = piece.strip()
                if not t:
                    continue
                counts[umbrella_label(t)] += 1

freq = {k: v for k, v in counts.items() if v > 0}
wc = WordCloud(width=920, height=520, background_color="white", colormap="viridis").generate_from_frequencies(
    freq,
)
plt.figure(figsize=(12, 7))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Umbrella themes from PROJECT_TERMS (keyword rules; not from per-term vectors)")
plt.tight_layout()
plt.show()

print("Top umbrella counts:", counts.most_common(20))

ImportError: Run: pip install wordcloud

### Auto-route **all** `;`-split terms (~49k) → a few word-cloud categories (Health, Engineering, …)

Keyword rules cannot cover tens of thousands of distinct strings. This cell instead:

1. **Counts** every term across `data/*_data.csv` (same split as before).
2. Loads **`sentence-transformers`** (same family as `export_embeddings.py` / `api/embeddings.py`).
3. Encodes a short **anchor paragraph per category** (tweak `CATEGORY_ANCHORS` to match what you mean by *Health*, *Engineering*, etc.).
4. Encodes **every unique term** in batches and assigns **nearest anchor by cosine similarity** (vectors are L2-normalized).
5. Rolls term counts up into **category totals** and draws a **word cloud** over those few labels (size = how much corpus mass landed in each theme).

At the end it writes **`project_term_theme_counts.json`** next to this notebook — the **dashboard** loads that file via **`GET /analytics/project-term-theme-cloud`** (restart backend after regenerating).

Optional: raise `MIN_TERM_COUNT` to drop ultra-rare strings before encoding. First run may **download the model** and take **CPU time** on ~49k phrases.

Requires: `pip install sentence-transformers` (and PyTorch). Uses `backend/` on `sys.path` for `api.embeddings.get_model`.

In [ ]:
from __future__ import annotations

import sys
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def resolve_indexer_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates: list[Path] = [cwd, *cwd.parents]
    for root in (cwd, cwd.parent):
        candidates.extend(
            [
                root / "backend" / "indexer",
                root / "indexer",
            ],
        )
    seen: set[Path] = set()
    for base in candidates:
        try:
            b = base.resolve()
        except OSError:
            continue
        if b in seen:
            continue
        seen.add(b)
        if (b / "proj_data_analysis.ipynb").is_file():
            return b
    return cwd


INDEXER = resolve_indexer_dir()
BACKEND_ROOT = INDEXER.parent
DATA_DIR = INDEXER / "data"
SPLIT_ON = ";"
CHUNK_ROWS = 50_000
MIN_TERM_COUNT = 1  # raise to 2–5 to skip ultra-rare noise before embedding
MAX_TERM_CHARS = 480  # trim long NIH phrases for the encoder
TERM_BATCH = 128
LOW_CONFIDENCE = 0.18  # max cosine below this → "Low_confidence"

# Short paragraphs — edit to match how you want the model to interpret each bucket.
CATEGORY_ANCHORS: dict[str, str] = {
    "Health": (
        "Biomedical and clinical research: human disease, patients, hospitals, public health, "
        "epidemiology, nutrition, mental health, physiology, pharmacology, cancer, immunology, "
        "infectious disease, neuroscience, aging, exercise, biomarkers, clinical trials."
    ),
    "Engineering": (
        "Engineering and applied technology: mechanical, electrical, civil, materials, chemical, "
        "aerospace, manufacturing, robotics, sensors, control systems, design, optimization, "
        "fabrication, hardware, energy systems, nanotechnology, MEMS, infrastructure."
    ),
    "Computing_AI": (
        "Computer science, software, algorithms, machine learning, artificial intelligence, "
        "data science, databases, cybersecurity, visualization, high performance computing, "
        "natural language processing, computer vision, statistical modeling."
    ),
    "Life_sci_basic": (
        "Basic life sciences in cells and molecules: genetics, genomics, proteomics, biochemistry, "
        "cell biology, microbiology, developmental biology, molecular mechanisms, model organisms, "
        "structural biology, bioinformatics without clinical focus."
    ),
    "Social_Education": (
        "Social, behavioral, economic, and education research: psychology, sociology, policy, "
        "inequality, schools, learning sciences, workforce training, surveys, implementation science."
    ),
    "Environment_Earth": (
        "Environmental, climate, ecology, ocean, atmospheric, geology, hydrology, sustainability, "
        "agriculture-environment interface, natural resources, conservation."
    ),
}

# --- 1) corpus frequencies for every ;-split term ---
paths = sorted(p.resolve() for p in DATA_DIR.glob("*_data.csv") if p.is_file())
if not paths:
    raise FileNotFoundError(f"No *_data.csv under {DATA_DIR.resolve()}")

term_freq: Counter[str] = Counter()
for path in paths:
    for chunk in pd.read_csv(
        path,
        usecols=["PROJECT_TERMS"],
        chunksize=CHUNK_ROWS,
        dtype=str,
        low_memory=False,
        on_bad_lines="skip",
    ):
        for raw in chunk["PROJECT_TERMS"].dropna():
            for piece in str(raw).split(SPLIT_ON):
                t = piece.strip()
                if t:
                    term_freq[t] += 1

terms_sorted = [t for t, c in term_freq.most_common() if c >= MIN_TERM_COUNT]
print(f"Unique terms (>= {MIN_TERM_COUNT} hits): {len(terms_sorted):,}  |  total token mentions: {sum(term_freq.values()):,}")

# --- 2) embed anchors + all terms; assign nearest category ---
sys.path.insert(0, str(BACKEND_ROOT))
from api.embeddings import get_model, get_model_name  # noqa: E402

try:
    from wordcloud import WordCloud
except ImportError as exc:
    raise ImportError("pip install wordcloud") from exc

labels = list(CATEGORY_ANCHORS.keys())
anchor_texts = list(CATEGORY_ANCHORS.values())

model = get_model()
print("Model:", get_model_name())

t0 = time.perf_counter()
A = np.asarray(
    model.encode(
        anchor_texts,
        batch_size=8,
        normalize_embeddings=True,
        show_progress_bar=False,
    ),
    dtype=np.float32,
)
blocks: list[np.ndarray] = []
for i in range(0, len(terms_sorted), TERM_BATCH):
    batch = [s[:MAX_TERM_CHARS] for s in terms_sorted[i : i + TERM_BATCH]]
    blocks.append(
        np.asarray(
            model.encode(
                batch,
                batch_size=TERM_BATCH,
                normalize_embeddings=True,
                show_progress_bar=False,
            ),
            dtype=np.float32,
        ),
    )
T = np.vstack(blocks)
elapsed = time.perf_counter() - t0
print(f"Encoded {len(terms_sorted):,} terms + {len(labels)} anchors in {elapsed:.1f}s")

sim = T @ A.T  # cosine since rows are unit-norm
best_i = sim.argmax(axis=1)
best_s = sim[np.arange(sim.shape[0]), best_i]

cat_mass: Counter[str] = Counter()
rows: list[dict[str, object]] = []
for term, j, sc in zip(terms_sorted, best_i, best_s):
    cat = labels[int(j)] if float(sc) >= LOW_CONFIDENCE else "Low_confidence"
    w = term_freq[term]
    cat_mass[cat] += w
    rows.append({"term": term, "category": cat, "cosine": float(sc), "count": w})

# Save full mapping for QA / tuning anchors
out_map = INDEXER / "term_to_category_by_embedding.csv"
pd.DataFrame(rows).sort_values("count", ascending=False).to_csv(out_map, index=False)
print(f"Wrote term→category table: {out_map.resolve()}")

# --- 3) word cloud over category masses ---
freq = {k: int(v) for k, v in cat_mass.items() if v > 0}
wc = WordCloud(width=960, height=540, background_color="white", colormap="magma").generate_from_frequencies(
    freq,
)
plt.figure(figsize=(12, 7))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Corpus mass by theme (each ;-split term → nearest anchor paragraph)")
plt.tight_layout()
plt.show()

print("Category masses (sum of term hit counts):", cat_mass.most_common())

# --- 4) JSON for FastAPI dashboard (`GET /analytics/project-term-theme-cloud`) ---
import json
from datetime import datetime, timezone

theme_json_path = INDEXER / "project_term_theme_counts.json"
theme_payload = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "method": "embedding_nearest_anchor_v1",
    "low_confidence_cosine": LOW_CONFIDENCE,
    "buckets": [
        {"label": str(k), "weight": int(v)}
        for k, v in sorted(cat_mass.items(), key=lambda kv: -kv[1])
    ],
}
theme_json_path.write_text(json.dumps(theme_payload, indent=2), encoding="utf-8")
print(f"Wrote dashboard JSON: {theme_json_path.resolve()}")

In [26]:
# Export merged data to a downloadable CSV
from pathlib import Path

output_path = Path("../merged_project_data.csv")
df_merged.to_csv(output_path, index=False)

print(f"Saved CSV: {output_path.resolve()}")
print(f"Rows: {len(df_merged):,}")

Saved CSV: /Users/Student/Documents/VSCODE/Sprinternship/backend/merged_project_data.csv
Rows: 75,995


In [ ]:
# Correlation heatmap using ALL df_merged columns
# Strategy: keep numeric columns as-is, encode non-numeric columns with integer codes.
# This keeps one feature per original column (no one-hot explosion).

sample_n = 15000
if len(df_merged) > sample_n:
    corr_input = df_merged.sample(sample_n, random_state=42).copy()
else:
    corr_input = df_merged.copy()

df_for_corr = pd.DataFrame(index=corr_input.index)

for col in corr_input.columns:
    numeric_col = pd.to_numeric(corr_input[col], errors="coerce")

    # If column has real numeric content, use numeric values.
    if numeric_col.notna().sum() > 0:
        df_for_corr[col] = numeric_col.astype("float32")
    else:
        # Otherwise encode categories/text into integer codes.
        # Missing values stay as NaN.
        codes = pd.factorize(corr_input[col].astype("string"), sort=True)[0]
        encoded = pd.Series(codes, index=corr_input.index).replace(-1, pd.NA)
        df_for_corr[col] = pd.to_numeric(encoded, errors="coerce").astype("float32")

# Drop columns with too few valid values to correlate meaningfully
df_for_corr = df_for_corr.dropna(axis=1, thresh=max(50, int(0.05 * len(df_for_corr))))

corr = df_for_corr.corr(numeric_only=True)

# Optional: plotting all columns can be unreadable, so cap displayed columns
max_cols = 46
if corr.shape[0] > max_cols:
    keep_cols = df_for_corr.var(numeric_only=True).sort_values(ascending=False).head(max_cols).index
    corr = df_for_corr[keep_cols].corr(numeric_only=True)

plt.figure(figsize=(15, 13))
sns.heatmap(corr, cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Correlation Heatmap (All df_merged Columns Encoded)")
plt.tight_layout()
plt.show()


Organizing 2020 Data


In [15]:
DATA_PATH2020 = Path("Project cv's/2020_ProjectData.csv")
DATA_PATH2020_ABSTRACT = Path("Project cv's/2020abs_projectData.csv")
if not DATA_PATH2020.exists():
    raise FileNotFoundError(f"Could not find data file at: {DATA_PATH2020.resolve()}")

df_project2020 = pd.read_csv(DATA_PATH2020)
df_abstract2020 = pd.read_csv(DATA_PATH2020_ABSTRACT, encoding="latin-1")

/var/folders/z1/x2sy0m8s59b3524k6xj71cxh0000gn/T/ipykernel_93101/2316930001.py:6: DtypeWarning: Columns (0: NIH_SPENDING_CATS, 1: PROJECT_TERMS) have mixed types. Specify dtype option on import or set low_memory=False.
  df_project2020 = pd.read_csv(DATA_PATH2020)


In [16]:
keep_cols = [
    "APPLICATION_ID",
    "PROJECT_TITLE",
    "PI_NAMEs",
    "ORG_NAME",
    "ORG_CITY",
    "ORG_STATE",
    "ORG_ZIPCODE",
    "ORG_COUNTRY",
    "IC_NAME",
    "FY",
    "PROJECT_START",
    "PROJECT_END",
    "TOTAL_COST",
    "ACTIVITY",
    "PROJECT_TERMS",
]

missing_cols = [c for c in keep_cols if c not in df_project2020.columns]
if missing_cols:
    print("These requested columns were not found:", missing_cols)

df_project2020 = df_project2020[[c for c in keep_cols if c in df_project2020.columns]].copy()
print(f"DataFrame shape after column filter: {df_project2020.shape}")
df_project2020.head()

DataFrame shape after column filter: (82428, 15)


,APPLICATION_ID,PROJECT_TITLE,PI_NAMEs,ORG_NAME,ORG_CITY,ORG_STATE,ORG_ZIPCODE,ORG_COUNTRY,IC_NAME,FY,PROJECT_START,PROJECT_END,TOTAL_COST,ACTIVITY,PROJECT_TERMS
0,8676197,Improving Care for PTSD,"SHINER, BRIAN (contact)",WHITE RIVER JUNCTION VA MEDICAL CENTER,White River Junction,VT,050093833,UNITED STATES,Veterans Affairs,2020,2014-07-01,2019-06-30,NaN,IK2,Affect;Area;Award;career;career development;Ca...
1,8785548,Design and Evaluation of User Centered Electro...,"AGHA, ZIA (contact)",VA SAN DIEGO HEALTHCARE SYSTEM,SAN DIEGO,CA,921610002,UNITED STATES,Veterans Affairs,2020,2015-10-01,2019-09-30,NaN,I01,acronyms;Address;Affect;Archives;Area;Automati...
2,8867710,Comparative Effectiveness of Delivery Methods ...,"MAVANDADI, SHAHRZAD (contact);WRAY, LAURA ODELL,",PHILADELPHIA VA MEDICAL CENTER,PHILADELPHIA,PA,191044551,UNITED STATES,Veterans Affairs,2020,2015-10-01,2020-05-31,NaN,I01,18 year old;Address;Adult;Adult Children;Age;A...
3,8900803,Relational Agent to Improve Alcohol Screening ...,"SIMON, STEVEN R. (contact)",VA BOSTON HEALTH CARE SYSTEM,BOSTON,MA,021304817,UNITED STATES,Veterans Affairs,2020,2014-07-01,2019-03-31,NaN,I01,Acute;Address;Age;Alcohol abuse;alcohol abuse ...
4,8978569,Assessing Hypertension Care for Aged Veterans:...,"MIN, LILLIAN CHIANG (contact)",VETERANS HEALTH ADMINISTRATION,ANN ARBOR,MI,481052303,UNITED STATES,Veterans Affairs,2020,2015-10-01,2019-12-31,NaN,I01,acute stroke;Adherence;Affect;Age;age group;ag...


In [19]:
df2020_merged = pd.merge(
    df_project2020,
    df_abstract2020,
    on="APPLICATION_ID",
    how="left", 
)

print(f"Matched rows: {len(df2020_merged):,}")
df2020_merged.head()

Matched rows: 82,428


,APPLICATION_ID,PROJECT_TITLE,PI_NAMEs,ORG_NAME,ORG_CITY,ORG_STATE,ORG_ZIPCODE,ORG_COUNTRY,IC_NAME,FY,PROJECT_START,PROJECT_END,TOTAL_COST,ACTIVITY,PROJECT_TERMS,ABSTRACT_TEXT
0,8676197,Improving Care for PTSD,"SHINER, BRIAN (contact)",WHITE RIVER JUNCTION VA MEDICAL CENTER,White River Junction,VT,050093833,UNITED STATES,Veterans Affairs,2020,2014-07-01,2019-06-30,NaN,IK2,Affect;Area;Award;career;career development;Ca...,DESCRIPTION (provided by applicant): BACK...
1,8785548,Design and Evaluation of User Centered Electro...,"AGHA, ZIA (contact)",VA SAN DIEGO HEALTHCARE SYSTEM,SAN DIEGO,CA,921610002,UNITED STATES,Veterans Affairs,2020,2015-10-01,2019-09-30,NaN,I01,acronyms;Address;Affect;Archives;Area;Automati...,DESCRIPTION (provided by applicant): To d...
2,8867710,Comparative Effectiveness of Delivery Methods ...,"MAVANDADI, SHAHRZAD (contact);WRAY, LAURA ODELL,",PHILADELPHIA VA MEDICAL CENTER,PHILADELPHIA,PA,191044551,UNITED STATES,Veterans Affairs,2020,2015-10-01,2020-05-31,NaN,I01,18 year old;Address;Adult;Adult Children;Age;A...,? DESCRIPTION (provided by applicant): ...
3,8900803,Relational Agent to Improve Alcohol Screening ...,"SIMON, STEVEN R. (contact)",VA BOSTON HEALTH CARE SYSTEM,BOSTON,MA,021304817,UNITED STATES,Veterans Affairs,2020,2014-07-01,2019-03-31,NaN,I01,Acute;Address;Age;Alcohol abuse;alcohol abuse ...,Background: VA has demonstrated national leade...
4,8978569,Assessing Hypertension Care for Aged Veterans:...,"MIN, LILLIAN CHIANG (contact)",VETERANS HEALTH ADMINISTRATION,ANN ARBOR,MI,481052303,UNITED STATES,Veterans Affairs,2020,2015-10-01,2019-12-31,NaN,I01,acute stroke;Adherence;Affect;Age;age group;ag...,? DESCRIPTION (provided by applicant): ...


In [ ]:
from pathlib import Path

output_path = Path("Project cv's/2020_PROJECT.csv")
df2020_merged.to_csv(output_path, index=False)

print(f"Saved CSV: {output_path.resolve()}")
print(f"Rows: {len(df2020_merged):,}")

ORGANIZE 2021 DATA

In [3]:
DATA_PATH2021 = Path("Project cv's/2021_ProjectData.csv")
DATA_PATH2021_ABSTRACT = Path("Project cv's/2021abs_projectData.csv")
if not DATA_PATH2021.exists():
    raise FileNotFoundError(f"Could not find data file at: {DATA_PATH2020.resolve()}")

df_project2021 = pd.read_csv(DATA_PATH2021)
df_abstract2021 = pd.read_csv(DATA_PATH2021_ABSTRACT, encoding="latin-1")

df_project2021.shape

(82940, 46)

In [4]:
keep_cols = [
    "APPLICATION_ID",
    "PROJECT_TITLE",
    "PI_NAMEs",
    "ORG_NAME",
    "ORG_CITY",
    "ORG_STATE",
    "ORG_ZIPCODE",
    "ORG_COUNTRY",
    "IC_NAME",
    "FY",
    "PROJECT_START",
    "PROJECT_END",
    "TOTAL_COST",
    "ACTIVITY",
    "PROJECT_TERMS",
]

missing_cols = [c for c in keep_cols if c not in df_project2021.columns]
if missing_cols:
    print("These requested columns were not found:", missing_cols)

df_project2021 = df_project2021[[c for c in keep_cols if c in df_project2021.columns]].copy()
print(f"DataFrame shape after column filter: {df_project2021.shape}")
df_project2021.head()

DataFrame shape after column filter: (82940, 15)


,APPLICATION_ID,PROJECT_TITLE,PI_NAMEs,ORG_NAME,ORG_CITY,ORG_STATE,ORG_ZIPCODE,ORG_COUNTRY,IC_NAME,FY,PROJECT_START,PROJECT_END,TOTAL_COST,ACTIVITY,PROJECT_TERMS
0,8836417,Using Peer Mentors to Support PACT Team Effort...,"LONG, JUDITH A (contact)",PHILADELPHIA VA MEDICAL CENTER,PHILADELPHIA,PA,191044551,UNITED STATES,Veterans Affairs,2021,2014-04-01,2018-03-31,NaN,I01,Address;Adult;African American;Ambulatory Care...
1,8982322,A Technology-Assisted Care Transition Interven...,"HOGAN, TIMOTHY PATRICK (contact)",EDITH NOURSE ROGERS MEMORIAL VETERANS HOSPITAL,BEDFORD,MA,017301114,UNITED STATES,Veterans Affairs,2021,2015-06-01,2021-06-30,NaN,I01,Admission activity;Adult;Adverse event;adverse...
2,9014369,QUERI-Office of Health Equity (OHE) Partnered ...,"WASHINGTON, DONNA (contact)",VA GREATER LOS ANGELES HEALTHCARE SYSTEM,LOS ANGELES,CA,900731003,UNITED STATES,Veterans Affairs,2021,2015-10-01,2025-09-30,NaN,I50,Address;Affect;Age;age group;Ambulatory Care F...
3,9063308,Building implementation science for VA healthc...,"SAFDAR, NASIA (contact)",WM S. MIDDLETON MEMORIAL VETERANS HOSP,MADISON,WI,537052254,UNITED STATES,Veterans Affairs,2021,2015-12-01,2017-11-30,NaN,I50,acute care;Address;Adoption;Advisory Committee...
4,9064929,Disseminating a Dashboard for VA Purchased Com...,"RUDOLPH, JAMES L (contact)",VA BOSTON HEALTH CARE SYSTEM,BOSTON,MA,021304817,UNITED STATES,Veterans Affairs,2021,2015-10-01,2017-09-30,NaN,I50,Address;Admission activity;Antipsychotic Agent...


In [5]:
df2021_merged = pd.merge(
    df_project2021,
    df_abstract2021,
    on="APPLICATION_ID",
    how="left", 
)

print(f"Matched rows: {len(df2021_merged):,}")
df2021_merged.head()

Matched rows: 82,940


,APPLICATION_ID,PROJECT_TITLE,PI_NAMEs,ORG_NAME,ORG_CITY,ORG_STATE,ORG_ZIPCODE,ORG_COUNTRY,IC_NAME,FY,PROJECT_START,PROJECT_END,TOTAL_COST,ACTIVITY,PROJECT_TERMS,ABSTRACT_TEXT
0,8836417,Using Peer Mentors to Support PACT Team Effort...,"LONG, JUDITH A (contact)",PHILADELPHIA VA MEDICAL CENTER,PHILADELPHIA,PA,191044551,UNITED STATES,Veterans Affairs,2021,2014-04-01,2018-03-31,NaN,I01,Address;Adult;African American;Ambulatory Care...,Background: Using peer mentors to support heal...
1,8982322,A Technology-Assisted Care Transition Interven...,"HOGAN, TIMOTHY PATRICK (contact)",EDITH NOURSE ROGERS MEMORIAL VETERANS HOSPITAL,BEDFORD,MA,017301114,UNITED STATES,Veterans Affairs,2021,2015-06-01,2021-06-30,NaN,I01,Admission activity;Adult;Adverse event;adverse...,? DESCRIPTION (provided by applicant): ...
2,9014369,QUERI-Office of Health Equity (OHE) Partnered ...,"WASHINGTON, DONNA (contact)",VA GREATER LOS ANGELES HEALTHCARE SYSTEM,LOS ANGELES,CA,900731003,UNITED STATES,Veterans Affairs,2021,2015-10-01,2025-09-30,NaN,I50,Address;Affect;Age;age group;Ambulatory Care F...,Background/Rationale: Health disparities are d...
3,9063308,Building implementation science for VA healthc...,"SAFDAR, NASIA (contact)",WM S. MIDDLETON MEMORIAL VETERANS HOSP,MADISON,WI,537052254,UNITED STATES,Veterans Affairs,2021,2015-12-01,2017-11-30,NaN,I50,acute care;Address;Adoption;Advisory Committee...,? DESCRIPTION (provided by applicant): ...
4,9064929,Disseminating a Dashboard for VA Purchased Com...,"RUDOLPH, JAMES L (contact)",VA BOSTON HEALTH CARE SYSTEM,BOSTON,MA,021304817,UNITED STATES,Veterans Affairs,2021,2015-10-01,2017-09-30,NaN,I50,Address;Admission activity;Antipsychotic Agent...,? DESCRIPTION (provided by applicant): ...


In [7]:
from pathlib import Path

output_path = Path("Project cv's/2021_PROJECT.csv")
df2021_merged.to_csv(output_path, index=False)

print(f"Saved CSV: {output_path.resolve()}")
print(f"Rows: {len(df2021_merged):,}")

Saved CSV: /Users/Student/Documents/VSCODE/Sprinternship/backend/indexer/Project cv's/2021_PROJECT.csv
Rows: 82,940


ORGANIZING 2022 DATA

In [8]:
DATA_PATH2022 = Path("Project cv's/2022_ProjectData.csv")
DATA_PATH2022_ABSTRACT = Path("Project cv's/2022abs_projectData.csv")
if not DATA_PATH2022.exists():
    raise FileNotFoundError(f"Could not find data file at: {DATA_PATH2020.resolve()}")

df_project2022 = pd.read_csv(DATA_PATH2022)
df_abstract2022 = pd.read_csv(DATA_PATH2022_ABSTRACT, encoding="latin-1")

df_project2022.shape

(83877, 46)

In [9]:
keep_cols = [
    "APPLICATION_ID",
    "PROJECT_TITLE",
    "PI_NAMEs",
    "ORG_NAME",
    "ORG_CITY",
    "ORG_STATE",
    "ORG_ZIPCODE",
    "ORG_COUNTRY",
    "IC_NAME",
    "FY",
    "PROJECT_START",
    "PROJECT_END",
    "TOTAL_COST",
    "ACTIVITY",
    "PROJECT_TERMS",
]

missing_cols = [c for c in keep_cols if c not in df_project2022.columns]
if missing_cols:
    print("These requested columns were not found:", missing_cols)

df_project2022 = df_project2022[[c for c in keep_cols if c in df_project2022.columns]].copy()
print(f"DataFrame shape after column filter: {df_project2022.shape}")
df_project2022.head()

DataFrame shape after column filter: (83877, 15)


,APPLICATION_ID,PROJECT_TITLE,PI_NAMEs,ORG_NAME,ORG_CITY,ORG_STATE,ORG_ZIPCODE,ORG_COUNTRY,IC_NAME,FY,PROJECT_START,PROJECT_END,TOTAL_COST,ACTIVITY,PROJECT_TERMS
0,9564059,Randomized Prospective Phase II Clinical Trial...,"STEA, BALDASSARRE ;UNGER, EVAN CHARLES (contact)","NUVOX PHARMA, LLC",TUCSON,AZ,857196803,UNITED STATES,NATIONAL CANCER INSTITUTE,2022,2010-05-24,2024-02-29,1000000.0,R44,Adverse event;Affect;Aftercare;Age;Age-Years;a...
1,9647222,Elucidating the regulation of RNA methylation ...,"LEE, GINA (contact)",UNIVERSITY OF CALIFORNIA-IRVINE,IRVINE,CA,926970001,UNITED STATES,NATIONAL CANCER INSTITUTE,2022,2022-09-01,2025-08-31,182226.0,K22,5&apos; Untranslated Regions;Adenosine;Autopha...
2,9666208,Phase IIa Trial of a Selective Glucocorticoid ...,"NEYLAN, THOMAS C (contact)",VETERANS AFFAIRS MED CTR SAN FRANCISCO,SAN FRANCISCO,CA,941211545,UNITED STATES,Veterans Affairs,2022,2022-01-01,2024-12-31,NaN,I01,Abortifacient Agents;Adverse event;Affinity;Ag...
3,9697389,The Pancreatic Cancer Microenvironment,"KANG, RUI (contact)",UT SOUTHWESTERN MEDICAL CENTER,DALLAS,TX,753909105,UNITED STATES,NATIONAL CANCER INSTITUTE,2022,2017-06-08,2024-03-31,364845.0,R01,Ablation;Affect;Alzheimer&apos;s Disease;Anima...
4,9717691,Mechanisms and functions of fatty acid oxidati...,"XIONG, JIANHUA (contact)",JOHNS HOPKINS UNIVERSITY,BALTIMORE,MD,212182680,UNITED STATES,"NATIONAL HEART, LUNG, AND BLOOD INSTITUTE",2022,2022-01-01,2024-12-31,249000.0,K22,academic preparation;Acetates;Acetyl Coenzyme ...


In [10]:
df2022_merged = pd.merge(
    df_project2022,
    df_abstract2022,
    on="APPLICATION_ID",
    how="left", 
)

print(f"Matched rows: {len(df2022_merged):,}")
df2022_merged.head()

Matched rows: 83,877


,APPLICATION_ID,PROJECT_TITLE,PI_NAMEs,ORG_NAME,ORG_CITY,ORG_STATE,ORG_ZIPCODE,ORG_COUNTRY,IC_NAME,FY,PROJECT_START,PROJECT_END,TOTAL_COST,ACTIVITY,PROJECT_TERMS,ABSTRACT_TEXT
0,9564059,Randomized Prospective Phase II Clinical Trial...,"STEA, BALDASSARRE ;UNGER, EVAN CHARLES (contact)","NUVOX PHARMA, LLC",TUCSON,AZ,857196803,UNITED STATES,NATIONAL CANCER INSTITUTE,2022,2010-05-24,2024-02-29,1000000.0,R44,Adverse event;Affect;Aftercare;Age;Age-Years;a...,Glioblastoma multiforme (GBM) is a malignant p...
1,9647222,Elucidating the regulation of RNA methylation ...,"LEE, GINA (contact)",UNIVERSITY OF CALIFORNIA-IRVINE,IRVINE,CA,926970001,UNITED STATES,NATIONAL CANCER INSTITUTE,2022,2022-09-01,2025-08-31,182226.0,K22,5&apos; Untranslated Regions;Adenosine;Autopha...,PROJECT SUMMARY/ABSTRACT\nOver the past decade...
2,9666208,Phase IIa Trial of a Selective Glucocorticoid ...,"NEYLAN, THOMAS C (contact)",VETERANS AFFAIRS MED CTR SAN FRANCISCO,SAN FRANCISCO,CA,941211545,UNITED STATES,Veterans Affairs,2022,2022-01-01,2024-12-31,NaN,I01,Abortifacient Agents;Adverse event;Affinity;Ag...,Posttraumatic stress disorder (PTSD) is a seri...
3,9697389,The Pancreatic Cancer Microenvironment,"KANG, RUI (contact)",UT SOUTHWESTERN MEDICAL CENTER,DALLAS,TX,753909105,UNITED STATES,NATIONAL CANCER INSTITUTE,2022,2017-06-08,2024-03-31,364845.0,R01,Ablation;Affect;Alzheimer&apos;s Disease;Anima...,Abstract\n The tumor microenvironment (TME) is...
4,9717691,Mechanisms and functions of fatty acid oxidati...,"XIONG, JIANHUA (contact)",JOHNS HOPKINS UNIVERSITY,BALTIMORE,MD,212182680,UNITED STATES,"NATIONAL HEART, LUNG, AND BLOOD INSTITUTE",2022,2022-01-01,2024-12-31,249000.0,K22,academic preparation;Acetates;Acetyl Coenzyme ...,Project Summary/Abstract\nThrough this K22 Car...


In [11]:
from pathlib import Path

output_path = Path("Project cv's/2022_PROJECT.csv")
df2022_merged.to_csv(output_path, index=False)

print(f"Saved CSV: {output_path.resolve()}")
print(f"Rows: {len(df2022_merged):,}")

Saved CSV: /Users/Student/Documents/VSCODE/Sprinternship/backend/indexer/Project cv's/2022_PROJECT.csv
Rows: 83,877


ORGANIZING 2023 DATA

In [12]:
DATA_PATH2023 = Path("Project cv's/2023_ProjectData.csv")
DATA_PATH2023_ABSTRACT = Path("Project cv's/2023abs_projectData.csv")
if not DATA_PATH2023.exists():
    raise FileNotFoundError(f"Could not find data file at: {DATA_PATH2023.resolve()}")

df_project2023 = pd.read_csv(DATA_PATH2023)
df_abstract2023 = pd.read_csv(DATA_PATH2023_ABSTRACT, encoding="latin-1")

df_project2023.shape

(85024, 46)

In [13]:
keep_cols = [
    "APPLICATION_ID",
    "PROJECT_TITLE",
    "PI_NAMEs",
    "ORG_NAME",
    "ORG_CITY",
    "ORG_STATE",
    "ORG_ZIPCODE",
    "ORG_COUNTRY",
    "IC_NAME",
    "FY",
    "PROJECT_START",
    "PROJECT_END",
    "TOTAL_COST",
    "ACTIVITY",
    "PROJECT_TERMS",
]

missing_cols = [c for c in keep_cols if c not in df_project2023.columns]
if missing_cols:
    print("These requested columns were not found:", missing_cols)

df_project2022 = df_project2022[[c for c in keep_cols if c in df_project2022.columns]].copy()
print(f"DataFrame shape after column filter: {df_project2023.shape}")
df_project2023.head()

DataFrame shape after column filter: (85024, 46)


,APPLICATION_ID,ACTIVITY,ADMINISTERING_IC,APPLICATION_TYPE,ARRA_FUNDED,AWARD_NOTICE_DATE,BUDGET_START,BUDGET_END,CFDA_CODE,CORE_PROJECT_NUM,...,SERIAL_NUMBER,STUDY_SECTION,STUDY_SECTION_NAME,SUBPROJECT_ID,SUFFIX,SUPPORT_YEAR,DIRECT_COST_AMT,INDIRECT_COST_AMT,TOTAL_COST,TOTAL_COST_SUB_PROJECT
0,8530931,I50,VA,1.0,N,2023-08-28,2012-10-01,2013-09-30,999.0,I50HX001175,...,1175,HQ4,HQ4 QUERI RRP Review Panel 3[HQ4],NaN,NaN,1.0,NaN,NaN,NaN,NaN
1,9075408,I50,VA,5.0,N,2023-09-06,2015-10-01,2016-09-30,999.0,I50HX001676,...,1676,HQ4,HQ4 QUERI RRP Review Panel 3[HQ4],NaN,NaN,2.0,NaN,NaN,NaN,NaN
2,9701510,R01,DK,7.0,N,2023-04-18,2023-01-01,2024-12-31,847.0,R01DK056123,...,56123,MCE,Molecular and Cellular Endocrinology Study Sec...,NaN,NaN,22.0,406000.0,263900.0,669900.0,NaN
3,9722309,R01,MH,5.0,N,2023-05-16,2023-05-01,2024-04-30,242.0,R01MH115504,...,115504,ZRG1,Special Emphasis Panel[ZRG1-BDA-A(50)R],NaN,NaN,3.0,442857.0,6237.0,449094.0,NaN
4,10003845,R01,FD,5.0,N,2023-08-29,2023-09-01,2024-08-31,103.0,R01FD006106,...,6106,ZFD1,ZFD1-SRC(99),NaN,NaN,3.0,339012.0,128763.0,467775.0,NaN


In [15]:
df2023_merged = pd.merge(
    df_project2023,
    df_abstract2023,
    on="APPLICATION_ID",
    how="left", 
)

print(f"Matched rows: {len(df2023_merged):,}")
df2023_merged.head()

Matched rows: 85,024


,APPLICATION_ID,ACTIVITY,ADMINISTERING_IC,APPLICATION_TYPE,ARRA_FUNDED,AWARD_NOTICE_DATE,BUDGET_START,BUDGET_END,CFDA_CODE,CORE_PROJECT_NUM,...,STUDY_SECTION,STUDY_SECTION_NAME,SUBPROJECT_ID,SUFFIX,SUPPORT_YEAR,DIRECT_COST_AMT,INDIRECT_COST_AMT,TOTAL_COST,TOTAL_COST_SUB_PROJECT,ABSTRACT_TEXT
0,8530931,I50,VA,1.0,N,2023-08-28,2012-10-01,2013-09-30,999.0,I50HX001175,...,HQ4,HQ4 QUERI RRP Review Panel 3[HQ4],NaN,NaN,1.0,NaN,NaN,NaN,NaN,\nBackground: Over 40 years ago Balint describ...
1,9075408,I50,VA,5.0,N,2023-09-06,2015-10-01,2016-09-30,999.0,I50HX001676,...,HQ4,HQ4 QUERI RRP Review Panel 3[HQ4],NaN,NaN,2.0,NaN,NaN,NaN,NaN,The Caregivers and Veterans Omnibus Health Ser...
2,9701510,R01,DK,7.0,N,2023-04-18,2023-01-01,2024-12-31,847.0,R01DK056123,...,MCE,Molecular and Cellular Endocrinology Study Sec...,NaN,NaN,22.0,406000.0,263900.0,669900.0,NaN,Thyroid hormone (TH) regulates numerous key ph...
3,9722309,R01,MH,5.0,N,2023-05-16,2023-05-01,2024-04-30,242.0,R01MH115504,...,ZRG1,Special Emphasis Panel[ZRG1-BDA-A(50)R],NaN,NaN,3.0,442857.0,6237.0,449094.0,NaN,DepressionÂ isÂ aÂ leadingÂ causeÂ ofÂ disabil...
4,10003845,R01,FD,5.0,N,2023-08-29,2023-09-01,2024-08-31,103.0,R01FD006106,...,ZFD1,ZFD1-SRC(99),NaN,NaN,3.0,339012.0,128763.0,467775.0,NaN,PROJECT SUMMARY\nThe proposed phase 2 multicen...


In [22]:
from pathlib import Path

output_path = Path("Project cv's/2023_PROJECT.csv")
df2023_merged.to_csv(output_path, index=False)

print(f"Saved CSV: {output_path.resolve()}")
print(f"Rows: {len(df2023_merged):,}")

Saved CSV: /Users/Student/Documents/VSCODE/Sprinternship/backend/indexer/Project cv's/2023_PROJECT.csv
Rows: 85,024


ORGANIZING 2024 DATA

In [17]:
DATA_PATH2024 = Path("Project cv's/2024_ProjectData.csv")
DATA_PATH2024_ABSTRACT = Path("Project cv's/2024abs_projectData.csv")
if not DATA_PATH2024.exists():
    raise FileNotFoundError(f"Could not find data file at: {DATA_PATH2024.resolve()}")

df_project2024 = pd.read_csv(DATA_PATH2024)
df_abstract2024 = pd.read_csv(DATA_PATH2024_ABSTRACT, encoding="latin-1")

df_project2024.shape

(83314, 46)

In [18]:
keep_cols = [
    "APPLICATION_ID",
    "PROJECT_TITLE",
    "PI_NAMEs",
    "ORG_NAME",
    "ORG_CITY",
    "ORG_STATE",
    "ORG_ZIPCODE",
    "ORG_COUNTRY",
    "IC_NAME",
    "FY",
    "PROJECT_START",
    "PROJECT_END",
    "TOTAL_COST",
    "ACTIVITY",
    "PROJECT_TERMS",
]

missing_cols = [c for c in keep_cols if c not in df_project2024.columns]
if missing_cols:
    print("These requested columns were not found:", missing_cols)

df_project2024 = df_project2024[[c for c in keep_cols if c in df_project2024.columns]].copy()
print(f"DataFrame shape after column filter: {df_project2024.shape}")
df_project2024.head()

DataFrame shape after column filter: (83314, 15)


,APPLICATION_ID,PROJECT_TITLE,PI_NAMEs,ORG_NAME,ORG_CITY,ORG_STATE,ORG_ZIPCODE,ORG_COUNTRY,IC_NAME,FY,PROJECT_START,PROJECT_END,TOTAL_COST,ACTIVITY,PROJECT_TERMS
0,9077077,Center for Innovation to Implementation (Ci2i)...,"ASCH, STEVEN M. (contact)",VETERANS ADMIN PALO ALTO HEALTH CARE SYS,PALO ALTO,CA,943041207,UNITED STATES,Veterans Affairs,2024,2013-10-01,2018-09-30,NaN,I50,Address;Area;Breeding;California;career develo...
1,9662896,Center for Care Delivery and Outcomes Research...,"FU, STEVEN (contact)",MINNEAPOLIS VA MEDICAL CENTER,MINNEAPOLIS,MN,554172309,UNITED STATES,Veterans Affairs,2024,2018-10-01,2023-09-30,NaN,I50,acronyms;Aging;Area;Cancer Control;cancer prev...
2,9663062,Center of Innovation for Complex Chronic Healt...,"WEAVER, FRANCES M. (contact)",EDWARD HINES JR VA HOSPITAL,HINES,IL,601413030,UNITED STATES,Veterans Affairs,2024,2018-10-01,2023-09-30,NaN,I50,Address;Antibiotics;Applications Grants;Appoin...
3,9663146,Center for Access & Delivery Research and Eval...,"PERENCEVICH, ELI N (contact)",IOWA CITY VA MEDICAL CENTER,IOWA CITY,IA,522462208,UNITED STATES,Veterans Affairs,2024,2018-10-01,2023-09-30,NaN,I50,Address;Area;barrier to care;Cardiovascular sy...
4,9663574,"Pain, Research, Informatics, Multimorbidities,...","BASTIAN, LORI A (contact)",VA CONNECTICUT HEALTHCARE SYSTEM,WEST HAVEN,CT,065162770,UNITED STATES,Veterans Affairs,2024,2018-10-01,2023-09-30,NaN,I50,Area;Award;behavioral health;Behavioral Scienc...


In [19]:
df2024_merged = pd.merge(
    df_project2024,
    df_abstract2024,
    on="APPLICATION_ID",
    how="left", 
)

print(f"Matched rows: {len(df2024_merged):,}")
df2024_merged.head()

Matched rows: 83,314


,APPLICATION_ID,PROJECT_TITLE,PI_NAMEs,ORG_NAME,ORG_CITY,ORG_STATE,ORG_ZIPCODE,ORG_COUNTRY,IC_NAME,FY,PROJECT_START,PROJECT_END,TOTAL_COST,ACTIVITY,PROJECT_TERMS,ABSTRACT_TEXT
0,9077077,Center for Innovation to Implementation (Ci2i)...,"ASCH, STEVEN M. (contact)",VETERANS ADMIN PALO ALTO HEALTH CARE SYS,PALO ALTO,CA,943041207,UNITED STATES,Veterans Affairs,2024,2013-10-01,2018-09-30,NaN,I50,Address;Area;Breeding;California;career develo...,DESCRIPTION (provided by applicant): \nThe ...
1,9662896,Center for Care Delivery and Outcomes Research...,"FU, STEVEN (contact)",MINNEAPOLIS VA MEDICAL CENTER,MINNEAPOLIS,MN,554172309,UNITED STATES,Veterans Affairs,2024,2018-10-01,2023-09-30,NaN,I50,acronyms;Aging;Area;Cancer Control;cancer prev...,The mission of the Center for Care Delivery an...
2,9663062,Center of Innovation for Complex Chronic Healt...,"WEAVER, FRANCES M. (contact)",EDWARD HINES JR VA HOSPITAL,HINES,IL,601413030,UNITED STATES,Veterans Affairs,2024,2018-10-01,2023-09-30,NaN,I50,Address;Antibiotics;Applications Grants;Appoin...,B. Abstract/Executive Summary: To create evide...
3,9663146,Center for Access & Delivery Research and Eval...,"PERENCEVICH, ELI N (contact)",IOWA CITY VA MEDICAL CENTER,IOWA CITY,IA,522462208,UNITED STATES,Veterans Affairs,2024,2018-10-01,2023-09-30,NaN,I50,Address;Area;barrier to care;Cardiovascular sy...,CADRE seeks renewed COIN funding to continue t...
4,9663574,"Pain, Research, Informatics, Multimorbidities,...","BASTIAN, LORI A (contact)",VA CONNECTICUT HEALTHCARE SYSTEM,WEST HAVEN,CT,065162770,UNITED STATES,Veterans Affairs,2024,2018-10-01,2023-09-30,NaN,I50,Area;Award;behavioral health;Behavioral Scienc...,This application proposes the continuation of ...


In [23]:
from pathlib import Path

output_path = Path("Project cv's/2024_PROJECT.csv")
df2024_merged.to_csv(output_path, index=False)

print(f"Saved CSV: {output_path.resolve()}")
print(f"Rows: {len(df2024_merged):,}")

Saved CSV: /Users/Student/Documents/VSCODE/Sprinternship/backend/indexer/Project cv's/2024_PROJECT.csv
Rows: 83,314
